In [1]:
import os
from PyPDF2 import PdfReader
import langchain

def extract_text_from_pdf(pdf_path: str) -> str:
    """
    Extracts text from all pages of a PDF file.

    Args:
        pdf_path (str): Path to the PDF file.

    Returns:
        str: Concatenated text from all pages.

    Raises:
        FileNotFoundError: If the file does not exist.
    """

    # Check if file exists
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"PDF file not found at path: {pdf_path}")

    # Read PDF
    reader = PdfReader(pdf_path)
    pages_text = []

    # Extract text from each page
    for page in reader.pages:
        text = page.extract_text()
        if text:
            pages_text.append(text.strip())

    # Join pages with newline and clean extra whitespace
    full_text = "\n".join(pages_text)
    return " ".join(full_text.split())

# if __name__=="__main__":
#     pdf_path = r"C:\Users\SHREE\Downloads\scorereport.pdf"
#     extracted_text = extract_text_from_pdf(pdf_path)
    
#     with open("scorereport.txt", "w") as f:
#         f.write(extracted_text)


In [2]:

from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> list[str]:
    """
    Splits text into chunks using LangChain's RecursiveCharacterTextSplitter.

    Args:
        text (str): Input text to split.
        chunk_size (int): Maximum size of each chunk (in characters).
        overlap (int): Overlap between chunks (in characters).

    Returns:
        list[str]: List of filtered text chunks (>= 30 characters).
    """

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", " "]
    )

    chunks = splitter.split_text(text)

    # Filter out chunks shorter than 30 characters
    return [chunk.strip() for chunk in chunks if chunk and len(chunk.strip()) >= 30]




c:\Users\SHREE\miniconda3\envs\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

from sentence_transformers import SentenceTransformer

# Module-level variable (lazy-loaded)
_model = None

def generate_embeddings(chunks: list[str]) -> list[list[float]]:
    """
    Generates embeddings for a list of text chunks using SentenceTransformer.

    Args:
        chunks (list[str]): List of text chunks.

    Returns:
        list[list[float]]: List of embeddings (each embedding is a list of floats).
    """
    global _model

    # Handle empty input
    if not chunks:
        return []

    # Lazy load the model (only once)
    if _model is None:
        _model = SentenceTransformer("all-MiniLM-L6-v2")

    # Generate embeddings
    embeddings = _model.encode(chunks, show_progress_bar=False)

    # Convert to plain Python list of lists
    return embeddings.tolist()

In [4]:
import os
import json
import faiss
import numpy as np


def build_and_save_index(
    chunks: list[str],
    embeddings: list[list[float]],
    save_dir: str = "faiss_index"
) -> dict:
    """
    Builds a FAISS IndexFlatL2 from embeddings and saves the index + chunks.

    Args:
        chunks (list[str]): List of text chunks.
        embeddings (list[list[float]]): Corresponding embeddings.
        save_dir (str): Directory to save index and chunks.

    Returns:
        dict: {"index_size": len(chunks), "save_dir": save_dir}
    """

    if len(chunks) != len(embeddings):
        raise ValueError("Chunks and embeddings must have the same length.")

    if not chunks:
        raise ValueError("Chunks list is empty.")

    # Ensure directory exists
    os.makedirs(save_dir, exist_ok=True)

    # Convert embeddings to numpy array (float32 required by FAISS)
    embeddings_np = np.array(embeddings).astype("float32")

    # Create FAISS index
    dimension = embeddings_np.shape[1]
    index = faiss.IndexFlatL2(dimension)

    # Add embeddings to index
    index.add(embeddings_np)

    # Save FAISS index
    faiss.write_index(index, os.path.join(save_dir, "index.faiss"))

    # Save chunks as JSON
    with open(os.path.join(save_dir, "chunks.json"), "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)

    return {"index_size": len(chunks), "save_dir": save_dir}


def load_index(save_dir: str = "faiss_index") -> tuple[faiss.Index, list[str]]:
    """
    Loads a FAISS index and corresponding chunks.

    Args:
        save_dir (str): Directory where index and chunks are stored.

    Returns:
        tuple: (faiss_index, chunks_list)

    Raises:
        FileNotFoundError: If directory or required files do not exist.
    """

    if not os.path.exists(save_dir):
        raise FileNotFoundError(f"Directory not found: {save_dir}")

    index_path = os.path.join(save_dir, "index.faiss")
    chunks_path = os.path.join(save_dir, "chunks.json")

    if not os.path.exists(index_path) or not os.path.exists(chunks_path):
        raise FileNotFoundError(
            f"Required files not found in directory: {save_dir}"
        )

    # Load FAISS index
    index = faiss.read_index(index_path)

    # Load chunks
    with open(chunks_path, "r", encoding="utf-8") as f:
        chunks = json.load(f)

    return index, chunks

In [5]:
def ingest_document(pdf_path: str) -> dict:
    """
    Full ingestion pipeline:
    PDF -> text -> chunks -> embeddings -> FAISS index

    Args:
        pdf_path (str): Path to the PDF file

    Returns:
        dict: Status + metadata
    """
    try:
        # Step 1: Extract text
        text = extract_text_from_pdf(pdf_path)

        # Step 2: Chunk text
        chunks = chunk_text(text)

        # Step 3: Generate embeddings
        embeddings = generate_embeddings(chunks)

        # Step 4: Convert embeddings to numpy float32
        embeddings_np = np.array(embeddings).astype("float32")

        # Step 5: Build and save FAISS index
        build_and_save_index(chunks, embeddings_np)

        return {
            "chunks": len(chunks),
            "status": "success",
            "index_path": "faiss_index/"
        }

    except Exception as e:
        return {
            "status": "error",
            "message": str(e)
        }